In [1]:
%load_ext autoreload
%autoreload 2

import os
os.environ['QT_QPA_PLATFORMTHEME'] = 'gtk3'  # Usa tema GTK3 en lugar de GNOME
import warnings
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import MaxNLocator
import seaborn as sns
%matplotlib qt

import tools.particle_swarm_optimization as pso
import tools.particle_swarm_optimization_plot as pso_plt
import tools.peristimulus_time_histogram as psth

warnings.filterwarnings("ignore")


              -- N E S T --
  Copyright (C) 2004 The NEST Initiative

 Version: 3.9.0
 Built: Oct  2 2025 06:57:03

 This program is provided AS IS and comes with
 NO WARRANTY. See the file LICENSE for details.

 Problems or suggestions?
   Visit https://www.nest-simulator.org

 Type 'nest.help()' to find out more about NEST.



In [4]:
# PSTH carpeta
import json

folder_root = os.path.join(os.getcwd(), 'results/')
#folder_root = os.path.join(os.getcwd(), 'results/wagatsuma/')
folder_path = folder_root

for folder0 in os.listdir(folder_root):
    trial_path = os.path.join(folder_root, folder0)
    
    # for folder in os.listdir(folder_path):
        # trial_path = os.path.join(folder_path, folder)
        
    if not os.path.isdir(trial_path):
        continue

    with open(os.path.join(trial_path, 'sim_params.json'), 'r') as file:
        sim_dict = json.load(file)
        t_sim = sim_dict.get("t_sim")
    break

bin_length = [50, 500]
print(folder_path)

for l_bin in bin_length:
    
    psth.PSTH_folders_data(folder_path, l_bin) 

    if l_bin == 50:
        psth.PSTH_figure(folder_path, t_sim, l_bin, 0)
        psth.PSTH_figure(folder_path, t_sim, l_bin, 1)
        psth.PSTH_figure(folder_path, t_sim, l_bin, 2)
        psth.PSTH_figure(folder_path, t_sim, l_bin, 3)

        try:
            psth.PSTH_figure(folder_path, t_sim, l_bin, 4)
            psth.PSTH_figure(folder_path, t_sim, l_bin, 5)
        except:
            print('fail0')

        plt.close('all')

/home/samuel/Documentos/NEST/canonical_microcircuits/results/


In [14]:
# grafico abstract cncic 
from utils import helpers
import scipy.stats as stats
from itertools import combinations

folder_root = '/home/samuel/Documentos/PostDoc/conferences & articles/2026 ICNCE/sim/v22/'
folder_path = os.path.join(folder_root, 'n02 k02 crf20 ecrf')
l_bin = 500
t_sim = 1500
l_3 = int(t_sim/l_bin/3)
layer = '2/3'
MCC = ['v1a','v1d','v2a','v2b']
fold = ['00/','10/','20/','30/']
#fold = ['00/','10/','30/']
Colors = ['#08C9FF','#FFAA00','#2237FF','#FF6700','#00FF98','#FFDA00']
# Colors = ['#0CB0FF','#FFAA00','#00FF6F','#FF6700','#203AFF','#FFDA00']

for f in fold:

    cols = np.array(['folder', 'layer', 'type'])
    params = range(int(t_sim/l_bin))
    names = np.concatenate((cols, params), axis=None)
    
    data_mcc = {}

    path = os.path.join(os.getcwd(), folder_path+f)
    psth.PSTH_folders_data(path, l_bin) 
    psth_data = pd.read_csv(os.path.join(path, 'psth_'+str(l_bin)+'.csv'), header=None, names=names)
    
    for m in range(4):
        mcc = MCC[m]
        color = Colors[m]
        group = layer+mcc

        subset = psth_data[(psth_data['type'] == 'exc') & (psth_data['layer'].isin([group]))][names[3:]].values
        data_mcc[mcc] = []

        for i in range(10):
            data_mcc[mcc] = np.concatenate((data_mcc[mcc], subset[i][2*l_3:3*l_3]), axis = 0) 
    #print(data_mcc)
    ###################### figure

    for figs in [0,2,4]:
        fig, ax = plt.subplots(layout='constrained', figsize=(4,3), sharex=True)

        # Configurar Noto Sans global
        plt.rcParams.update({
            'font.family': 'sans-serif',
            'font.sans-serif': ['Noto Sans', 'DejaVu Sans', 'Arial', 'Helvetica'],
            'font.weight': 'bold',
            'axes.labelweight': 'bold',
            'axes.titleweight': 'bold',
        })

        position = [0, 1]
        if figs == 4:
            meanbar = [20, int(f[:-1])]
            stdbar = [0, 0]
        else:

            meanbar = [np.mean(data_mcc[MCC[figs]]), np.mean(data_mcc[MCC[figs+1]])]
            stdbar = [np.std(data_mcc[MCC[figs]]), np.std(data_mcc[MCC[figs+1]])]

        # strmean = np.round(meanbar, 2)
        # strstd = np.round(stdbar, 2)
        # n_str = ''
        # for i in range(5):
        #     n_str = n_str + str(strmean[i]) + ' $\pm\,$' + str(strstd[i]) + ' & '
        # print(n_str)

        plt.bar(position, meanbar, fc=Colors[figs:figs+2], ec='k', lw=3, capsize=5)
        plt.errorbar(position, meanbar, yerr=stdbar, fmt=' ', capsize=15, capthick=3, elinewidth=3, ecolor='k') # elinewidth y ecolor

        # Statistical test for each group in the section
        if figs != 4:
            h, sig = 0.5, False
            y =  max(meanbar)+max(stdbar) + h
            yy = y

            stat, pvalue = stats.wilcoxon(data_mcc[MCC[figs]], data_mcc[MCC[figs+1]])
            #print(pvalue)

            # Plot and pimp my error
            if pvalue <= 0.05:
                x1, x2 = 0, 1
                plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=3, c='k')
                if pvalue <= 0.005:
                    plt.text((x1+x2)*.5, y+h, '***', ha='center', va='bottom', color='k', size=20, weight='heavy')
                elif pvalue <= 0.01:
                    plt.text((x1+x2)*.5, y+h, '**', ha='center', va='bottom', color='k', size=20, weight='heavy')
                else:
                    plt.text((x1+x2)*.5, y+h, '*', ha='center', va='bottom', color='k', size=20, weight='heavy')
            
        if figs == 0:
            top = 11
        if figs == 2:
            top = 15
        if figs == 4:
            top = 32


        plt.ylim([0, top])

        #plt.xlabel('Time (ms)',fontsize=15)
        plt.ylabel('Firing rate (Hz)',fontsize=25)
        #plt.title('Firing Rate of Exc neurons in Layer '+layer,fontsize=22)
        plt.xticks([0, 1],['CRF', 'eCRF'])

        plt.xticks(fontsize = 25) 
        plt.yticks(fontsize = 20) 

        for spine in ax.spines.values():
            spine.set_linewidth(2.5) 
        plt.show()
        plt.savefig(os.path.join(folder_root, 'bar_'+str(figs)+'_'+f[:-1]), dpi=300)
        plt.close('all')  

In [44]:
# PSTH figuras varias curvas

folder_root = '/home/samuel/Documentos/PostDoc/conferences & articles/2026 ICNCE/sim/v22/'
folder_path = os.path.join(folder_root, 'n02 k02 crf20 ecrf30 2/')
t_sim = 1500
l_bin = 50

# add columns names
cols = np.array(['folder', 'layer', 'type'])
params = range(int(t_sim/l_bin))
names = np.concatenate((cols, params), axis=None)

csv_data = pd.read_csv(os.path.join(folder_path, 'mean_'+str(l_bin)+'.csv'), header=None, names=names)
bin_centers = np.linspace(l_bin/2, t_sim-(l_bin/2), int(t_sim/l_bin))

l_3 = int(t_sim/l_bin/3)
Layers = ['2/3']
MCC = ['v1a','v1d','v2a','v2b']

for m in [0]:

    for layer in Layers:
        group = layer+MCC[m]
        
        #subset = np.concatenate((subset[:, 3:4], subset[:, 1:]), axis=1)
        fig, ax = plt.subplots(layout='constrained', figsize=(5,3), sharex=True)

        # Configurar Noto Sans global
        plt.rcParams.update({
            'font.family': 'sans-serif',
            'font.sans-serif': ['Noto Sans', 'DejaVu Sans', 'Arial', 'Helvetica'],
            'font.weight': 'bold',
            'axes.labelweight': 'bold',
            'axes.titleweight': 'bold',
        })

        mean_subset = csv_data[(csv_data['folder'] == 'mean') & (csv_data['type'] == 'exc') & (csv_data['layer'].isin([layer+MCC[m+1]]))][names[3:]].values
        std_subset = csv_data[(csv_data['folder'] == 'std') & (csv_data['type'] == 'exc') & (csv_data['layer'].isin([layer+MCC[m+1]]))][names[3:]].values
        y, ci, bin_centers = np.squeeze(mean_subset)[:-10], np.squeeze(std_subset)[:-10], bin_centers[:-10]

        ax.plot(bin_centers, y, '.-',lw=5, color='#FFAA00', markersize=15)
        ax.fill_between(bin_centers, (y-ci), (y+ci), color='#FFAA00', alpha=.3, lw=0.1)

        mean_subset = csv_data[(csv_data['folder'] == 'mean') & (csv_data['type'] == 'exc') & (csv_data['layer'].isin([layer+MCC[m]]))][names[3:]].values
        std_subset = csv_data[(csv_data['folder'] == 'std') & (csv_data['type'] == 'exc') & (csv_data['layer'].isin([layer+MCC[m]]))][names[3:]].values
        y, ci = np.squeeze(mean_subset)[:-10], np.squeeze(std_subset)[:-10]

        ax.plot(bin_centers, y, '.-',lw=5, color='#08C9FF', markersize=15)
        ax.fill_between(bin_centers, (y-ci), (y+ci), color='#08C9FF', alpha=.3, lw=0.1)

        # y, ci= subset[0+s], subset[1+s]
        # ax.plot(bin_centers, y, '.-', lw=4, color='#1E4EFF', markersize=13)
        # ax.fill_between(bin_centers, (y-ci), (y+ci), color='#1E4EFF', alpha=.1)

        # y, ci= subset[4+s], subset[5+s]
        # ax.plot(bin_centers, y, '.-',lw=4, color='#FF6A00', markersize=13)
        # ax.fill_between(bin_centers, (y-ci), (y+ci), color='#FF6A00', alpha=.1)

        #################################################################################
        
        
        bot, top = 0, 10
        ax.plot([200, 200], [bot, top], 'k--', lw=3)
        ax.plot([600, 600], [bot, top], 'k--', lw=3)

        
        plt.xlabel('Time (ms)',fontsize=18)
        plt.ylabel('Firing rate (Hz)',fontsize=18)
        plt.title('PSTH neurons V1 Layer '+Layers[0],fontsize=22)
        plt.xticks(fontsize = 18) 
        plt.yticks(fontsize = 18) 
        plt.xticks([0, 250, 500, 750, 1000],[0, 250, 500, 750, 1000])
        plt.yticks([0, 3, 6, 9, 12],[0, 3, 6, 9, 12])
        plt.yticks([0, 2.5, 5, 7.5, 10],[0, 2.5, 5, 7.5, 10])

        for spine in ax.spines.values():
            spine.set_linewidth(2.5) 
            
        bot, top = -0.5, 10.5
        plt.ylim([bot, top])
        plt.show()
        plt.savefig(os.path.join(folder_path, 'psthComp_'), dpi=500)
        plt.close('all')  

        #continue
    

In [3]:
folder_root = '/home/samuel/Documentos/PostDoc/conferences & articles/2026 ICNCE/sim/v2/'
folder_path = os.path.join(folder_root, 'n02 k02 crf20 ecrf30 2/')
t_sim = 1000
l_bin = 50
bin_centers = np.linspace(l_bin/2, t_sim-(l_bin/2), int(t_sim/l_bin))

# Configurar Noto Sans global
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Noto Sans', 'DejaVu Sans', 'Arial', 'Helvetica'],
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold',
})

fig, ax = plt.subplots(layout='constrained', figsize=(5,2.5), sharex=True)

mean_subset = np.zeros(20)
mean_subset[12:] = 30
ax.plot(bin_centers, mean_subset, '.-',lw=5, color='#FFDA00', markersize=15)

mean_subset = np.zeros(20)
mean_subset[4:] = 20
ax.plot(bin_centers, mean_subset, '.-',lw=5, color='#00FF98', markersize=15)

bot, top = ax.get_ylim() 
print(bot)
print(top)
ax.plot([200, 200], [bot, 31.5], 'k--', lw=3)
ax.plot([600, 600], [bot, 31.5], 'k--', lw=3)

plt.ylabel('Firing rate (Hz)',fontsize=18)
plt.xlabel('Time (ms)',fontsize=18)
plt.title('Thalamic neuron stimulation',fontsize=22)
plt.xticks(fontsize = 18) 
plt.yticks(fontsize = 18) 
plt.xticks([0, 250, 500, 750, 1000],[0, 250, 500, 750, 1000])

for spine in ax.spines.values():
    spine.set_linewidth(2.5) 

plt.show()
plt.savefig(os.path.join(folder_path, 'tal_ecrf'), dpi=500)
plt.close('all')  

-1.5
31.5


In [43]:
folder_root = '/home/samuel/Documentos/PostDoc/conferences & articles/2026 ICNCE/sim/v2/'
folder_path = os.path.join(folder_root, 'n02 k02 crf20 ecrf30 2/')


# Configurar Noto Sans global
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Noto Sans', 'DejaVu Sans', 'Arial', 'Helvetica'],
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold',
})

fig, ax = plt.subplots(layout='constrained', figsize=(3, 2), sharex=True)

mean_subset = np.ones(10)*8
ax.plot(mean_subset, '.-',lw=5, color='#08C9FF', markersize=15)
ax.plot([-10,100],[0,0], '-', lw=1, color='#000000', markersize=20)

plt.ylabel('Firing\nrate (Hz)',fontsize=20)# labelpad=-5)
plt.xlabel('Time (ms)',fontsize=20)
plt.title('High activity',fontsize=22)
plt.xticks(fontsize = 18) 
plt.yticks(fontsize = 15) 
plt.ylim([-1,10])
plt.xlim([-1,10])

for spine in ax.spines.values():
    spine.set_linewidth(2.5) 

# Alternativa: ocultar completamente los ticks
ax.tick_params(axis='both', which='both', length=0)  # Oculta las marcas
ax.set_xticklabels([])
ax.set_yticklabels([])
# ax.set_yticks([0])  # Mantener los ticks en sus posiciones
# ax.set_yticklabels(['0'])

plt.show()
plt.savefig(os.path.join(folder_path, 'tal_ecrf'), dpi=500)
plt.close('all') 